# Studi Kasus: Simulasi Kebijakan Efisiensi Energi Bangunan (*Building Energy Policy Co-Simulation*) Berbasis HPC

## 1. Latar Belakang & Skenario Masalah

Sebuah institusi pendidikan di iklim tropis mengalami lonjakan biaya listrik akibat penggunaan pendingin udara (*AC*) dan perangkat elektronik yang tidak efisien. Pemodelan fisika bangunan tradisional (seperti *EnergyPlus*) hanya memperhitungkan struktur fisik bangunan dan sistem HVAC, tetapi tidak mampu mensimulasikan perilaku manusia (*occupant behavior*) di dalam bangunan.

**Solusi Berbasis HPC (*Co-Simulation*):**
Menggabungkan pemodelan fisika bangunan (*Energy Plus*) dengan **Agent-Based Modeling (ABM)** untuk mensimulasikan pergerakan dan aktivitas manusia di dalam setiap ruangan. Karena jumlah agen/manusia bisa mencapai ribuan dan interaksi terjadi secara *real-time*, proses komputasi membutuhkan daya komputasi tinggi (*High Performance Computing* / HPC) berbasis eksekusi paralel (multiprocessing / MPI) untuk mengevaluasi berbagai skenario kebijakan efisiensi energi.

---

## 2. Kebutuhan Sistem (Functional & Non-Functional)

* **Kebutuhan Fungsional**:
1. Sistem mensimulasikan konsumsi energi harian dari beberapa zona ruangan berdasarkan perilaku agen (*misal: jam masuk, pengaturan suhu AC*).
2. Sistem menjalankan simulasi secara paralel di beberapa *core* prosesor untuk mempercepat waktu komputasi (*scalable simulation*).
3. Sistem membandingkan skenario kebijakan (*baseline vs. efisiensi energi*) dan menghitung total penghematan energi.


* **Kebutuhan Non-Fungsional**:
1. Simulasi harus efisien dan dapat diskalakan (*high throughput*) menggunakan teknik pemrosesan paralel (*Multiprocessing / HPC-friendly Python*).



---

## 3. Implementasi Kode Python untuk Simulasi Efisiensi Energi (HPC Multiprocessing)

Berikut adalah kode Python yang mengimplementasikan simulasi paralel (*Agent-Based Energy Simulation*) menggunakan pustaka `multiprocessing` dan `pandas`:

In [ ]:
import multiprocessing as mp
import pandas as pd
import random
import time

# 1. Fungsi Simulasi Perilaku Agen & Konsumsi Energi Zona Bangunan (Fungsi Komputasi)
def simulate_zone_energy(zone_id, num_agents, policy_active=False):
    """
    Mensimulasikan konsumsi energi (kWh) untuk satu zona berdasarkan jumlah agen/manusia
    dan skenario kebijakan energi.
    """
    total_energy_kwh = 0.0
    hours = 24  # Simulasi 24 jam

    for hour in range(hours):
        # Pola kehadiran manusia (puncak aktivitas jam 08:00 - 17:00)
        if 8 <= hour <= 17:
            active_agents = int(num_agents * random.uniform(0.7, 1.0))
        else:
            active_agents = int(num_agents * random.uniform(0.05, 0.2))

        # Beban dasar bangunan (penerangan & sistem dasar)
        base_load = 5.0  # kWh

        # Beban per agen (perangkat elektronik + beban AC)
        agent_load = active_agents * 0.35  # kWh per orang

        # Pengaruh Kebijakan Efisiensi Energi (misal: batasan suhu AC min 25°C & matikan daya otomatis)
        if policy_active:
            # Penghematan energi sebesar 20% saat jam kerja dan matikan penuh di luar jam kerja
            if 8 <= hour <= 17:
                agent_load *= 0.80
            else:
                agent_load *= 0.30
                base_load *= 0.50

        hourly_consumption = base_load + agent_load
        total_energy_kwh += hourly_consumption

    return {
        'Zone_ID': f'Zone_{zone_id}',
        'Num_Agents': num_agents,
        'Policy_Active': policy_active,
        'Total_Energy_kWh': round(total_energy_kwh, 2)
    }

# 2. Eksekusi Simulasi Paralel Menggunakan HPC Multiprocessing
if __name__ == '__main__':
    start_time = time.time()

    # Menyiapkan data 8 zona bangunan dengan variasi populasi agen
    zones_data = [(i, random.randint(30, 150)) for i in range(1, 9)]

    print("--- Memulai Simulasi Paralel Berbasis HPC (Multiprocessing) ---")
    print(f"Jumlah Core CPU yang Tersedia: {mp.cpu_count()}\n")

    # Skenario 1: Tanpa Kebijakan Energi (Baseline)
    with mp.Pool(processes=mp.cpu_count()) as pool:
        baseline_results = pool.starmap(simulate_zone_energy, [(z_id, ag, False) for z_id, ag in zones_data])

    # Skenario 2: Dengan Kebijakan Efisiensi Energi (Energy Policy)
    with mp.Pool(processes=mp.cpu_count()) as pool:
        policy_results = pool.starmap(simulate_zone_energy, [(z_id, ag, True) for z_id, ag in zones_data])

    # 3. Pengolahan Data & Evaluasi Hasil Simulasi
    df_base = pd.DataFrame(baseline_results)
    df_pol = pd.DataFrame(policy_results)

    total_base_kwh = df_base['Total_Energy_kWh'].sum()
    total_pol_kwh = df_pol['Total_Energy_kWh'].sum()
    savings_percent = ((total_base_kwh - total_pol_kwh) / total_base_kwh) * 100

    print("--- Hasil Ringkasan Simulasi Energi ---")
    print(f"Total Konsumsi Energi Baseline (Tanpa Kebijakan): {total_base_kwh:.2f} kWh")
    print(f"Total Konsumsi Energi Efisiensi (Dengan Kebijakan): {total_pol_kwh:.2f} kWh")
    print(f"Estimasi Penghematan Energi: {savings_percent:.2f}%\n")

    execution_time = time.time() - start_time
    print(f"Waktu Eksekusi Komputasi Paralel: {execution_time:.4f} detik")

---

## 4. Evaluasi & Manfaat Pendekatan HPC

* **Persepatan Komputasi (*Speedup*)**: Dengan memanfaatkan arsitektur eksekusi paralel (*multiprocessing* / HPC *cluster*), simulasi ribuan interaksi agen dan kondisi fisik bangunan dapat dilakukan dalam hitungan detik dibandingkan eksekusi sekuensial tunggal.
* **Pengambilan Keputusan Berbasis Bukti (*Evidence-Based Policy*)**: Hasil simulasi membantu pengelola kampus/gedung membuat keputusan efisiensi energi yang realistis tanpa mengganggu kenyamanan occupants maupun kelangsungan riset penting.